# Notebook 12 – Feature Leakage

# **Topic 1 – What is Feature Leakage?**

## Step 1 – Understand the Concept

Feature Leakage occurs when a Machine Learning model receives information during training that would not be available when making real-world predictions.

This causes the model to learn using information that it should not have access to.

As a result, the model may show unusually high performance during training or testing but perform poorly in real-world situations.

Feature leakage can happen during:
- Feature Engineering
- Data Preprocessing
- Data Splitting
- Feature Selection
- Aggregation
- Target Encoding

### Example

Suppose a company wants to predict whether a customer will cancel a subscription.

The dataset contains:
- Customer Age
- Monthly Charges
- Contract Type
- Cancellation Date

If Cancellation Date is used to predict whether the customer will cancel, the model receives information about an outcome that has already happened.

This is Feature Leakage.

### Important Principle

A Machine Learning model should only use information that would genuinely be available at the prediction time.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

A student performance model predicts exam results using study hours.

If the model receives the final exam marks as an input feature, leakage occurs because the target information is already included.

### 2. Real-World Example

A hospital predicts whether a patient will develop a disease.

Using a medical test result collected after the diagnosis as an input feature creates leakage.

### 3. Business Example

An e-commerce company predicts whether an order will be returned.

Using the actual Return Date as an input feature creates leakage.

### 4. AI/ML Use Case

A customer churn prediction model uses Cancellation Status or Cancellation Date to predict customer churn.

The model may achieve misleadingly high accuracy.

### Why is this useful?

Understanding leakage helps prevent unrealistic model performance and improves real-world prediction reliability.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create customer dataset

np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "Customer_Age": np.random.randint(18, 65, n),
    "Monthly_Charges": np.random.randint(500, 5000, n),
    "Tenure_Months": np.random.randint(1, 60, n)
})

# Create target

df["Churn"] = (
    (df["Tenure_Months"] < 12) &
    (df["Monthly_Charges"] > 2500)
).astype(int)

# Incorrect feature engineering

df["Churn_Information"] = df["Churn"]

# Separate features and target

X_wrong = df[
    ["Customer_Age", "Monthly_Charges",
     "Tenure_Months", "Churn_Information"]
]

y = df["Churn"]

# Split data

X_train, X_test, y_train, y_test = train_test_split(
    X_wrong,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train model

model_wrong = RandomForestClassifier(
    random_state=42
)

model_wrong.fit(X_train, y_train)

predictions = model_wrong.predict(X_test)

print("Accuracy with Leakage:",
      accuracy_score(y_test, predictions))

Accuracy with Leakage: 1.0


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates a synthetic customer dataset.
2. Creates a churn target using customer information.
3. Copies the target into another column.
4. Uses the copied target as an input feature.
5. Trains a Random Forest model.
6. Evaluates the model using test data.

### Why was the feature created?

Churn_Information was intentionally created to demonstrate incorrect feature engineering.

### Source Columns

Churn.

### Feature Logic

Churn_Information = Churn

### Information Represented

The feature directly contains the actual churn outcome.

### How could it help Machine Learning?

It allows the model to access the answer directly, resulting in misleadingly high performance.

### Important Observation

The model's test accuracy is not trustworthy because the test feature also contains the test target.

This is not a valid predictive experiment.

## Step 4 – Justify Your Feature

### Feature Name
Churn_Information

### Source Columns
Churn.

### Feature Logic / Formula
Churn_Information = Churn.

### Reason for Creation
Created intentionally to demonstrate target leakage.

### Expected Relationship with Target
Perfect relationship because the feature directly copies the target.

### Potential Business Meaning
Represents the actual customer churn outcome.

### Potential ML Value
No valid predictive value because the answer is already provided.

### Leakage Risk
Extremely high. This feature directly reveals the target.

### Final Decision
REMOVE.

Never use actual target information as an input feature.

# **Topic 2 – Target Leakage**

## Step 1 – Understand the Concept

Target Leakage occurs when an input feature contains information that directly or indirectly reveals the target variable.

This usually happens when a feature is created using the target itself or information generated after the target outcome.

### Example

A company wants to predict whether a customer will churn.

Target:
Churn

Incorrect Feature:
Churn_Status_Copy

If Churn_Status_Copy contains the actual churn value, the model already knows the answer.

### Common Causes

- Copying the target into an input feature.
- Using future outcome information.
- Creating features using target-derived information.
- Including post-outcome columns.

### Important Principle

Input features must contain information available before the target outcome occurs.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

Predicting examination results using the final exam score as an input feature.

### 2. Real-World Example

Predicting disease occurrence using a test result obtained after diagnosis.

### 3. Business Example

Predicting loan default using the actual loan recovery status.

### 4. AI/ML Use Case

A fraud detection model uses a confirmed fraud investigation result to predict whether the transaction is fraudulent.

### Why is this useful?

Preventing target leakage ensures that the model learns from legitimate predictive information rather than already-known answers.

In [2]:
# Create a sample loan dataset

np.random.seed(42)

loan_df = pd.DataFrame({
    "Income": np.random.randint(15000, 100000, 500),
    "Credit_Score": np.random.randint(300, 850, 500),
    "Loan_Amount": np.random.randint(10000, 500000, 500)
})

# Create target

loan_df["Loan_Default"] = (
    (loan_df["Credit_Score"] < 550) &
    (loan_df["Loan_Amount"] > 250000)
).astype(int)

# Incorrect feature: target copied into input

loan_df["Default_Copy"] = loan_df["Loan_Default"]

# Identify suspicious feature

print(
    "Correlation with Target:",
    loan_df["Default_Copy"].corr(
        loan_df["Loan_Default"]
    )
)

# Remove leakage feature

correct_features = loan_df[
    ["Income", "Credit_Score", "Loan_Amount"]
]

target = loan_df["Loan_Default"]

print("Correct Input Features:")
display(correct_features.head())

Correlation with Target: 1.0
Correct Input Features:


,Income,Credit_Score,Loan_Amount
0,30795,572,183532
1,15860,361,100397
2,91820,641,82847
3,69886,696,103607
4,21265,318,230178


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates a synthetic loan dataset.
2. Generates a loan default target.
3. Copies the target into Default_Copy.
4. Calculates correlation with the target.
5. Removes the copied target from the correct input dataset.

### Incorrect Feature

Default_Copy.

### Source Columns

Loan_Default.

### Feature Logic

Default_Copy = Loan_Default.

### Information Represented

The actual loan default outcome.

### Correct Features

- Income
- Credit_Score
- Loan_Amount

These are intended to represent information available before the loan outcome.

### ML Usefulness

The correct features can help the model identify financial risk patterns without directly revealing the answer.

## Step 4 – Justify Your Feature

### Feature Name
Default_Copy

### Source Columns
Loan_Default.

### Feature Logic / Formula
Default_Copy = Loan_Default.

### Reason for Creation
To demonstrate direct target leakage.

### Expected Relationship with Target
Perfect correlation with Loan_Default.

### Potential Business Meaning
Represents the actual loan default result.

### Potential ML Value
No valid predictive value because it directly reveals the answer.

### Leakage Risk
Extremely high.

### Final Decision
REMOVE.

Retain Income, Credit_Score, and Loan_Amount as candidate input features, subject to availability at prediction time.

# **Topic 3 – Train-Test Leakage**

## Step 1 – Understand the Concept

Train-Test Leakage occurs when information from the testing dataset influences the training process.

The purpose of a test dataset is to evaluate how well a model performs on unseen data.

If preprocessing steps learn information from the complete dataset before splitting, the test dataset indirectly influences training.

### Common Causes

1. Applying StandardScaler before train-test splitting.
2. Applying PCA before splitting.
3. Performing feature selection on the complete dataset.
4. Imputing missing values using the entire dataset.
5. Selecting features based on test performance.

### Incorrect Approach

Fit preprocessing using the complete dataset.

Then split into training and testing data.

### Correct Approach

Split the dataset first.

Fit preprocessing using training data only.

Transform testing data using the fitted training preprocessing object.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

A student dataset is divided into training and testing sets.

If the average marks from both datasets are used to fill missing training values, test information influences training.

### 2. Real-World Example

A medical dataset uses all patient records to calculate scaling parameters before separating training and testing patients.

### 3. Business Example

A sales prediction model uses the mean sales value from future test records when preprocessing training data.

### 4. AI/ML Use Case

A PCA model is fitted on the complete dataset before train-test splitting.

The test distribution influences the learned principal components.

### Why is this useful?

It preserves an independent test dataset and provides a more realistic evaluation.

In [3]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Generate dataset

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    random_state=42
)

# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# INCORRECT APPROACH

scaler_wrong = StandardScaler()

X_all_scaled = scaler_wrong.fit_transform(X)

print("Incorrect: Scaler fitted on complete dataset")

# CORRECT APPROACH

scaler_correct = StandardScaler()

X_train_scaled = scaler_correct.fit_transform(
    X_train
)

X_test_scaled = scaler_correct.transform(
    X_test
)

print("Correct: Scaler fitted on training data only")

print("Training Shape:", X_train_scaled.shape)

print("Testing Shape:", X_test_scaled.shape)

Incorrect: Scaler fitted on complete dataset
Correct: Scaler fitted on training data only
Training Shape: (800, 5)
Testing Shape: (200, 5)


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates a classification dataset.
2. Splits it into training and testing data.
3. Demonstrates incorrect scaling on the complete dataset.
4. Demonstrates correct scaling on training data only.
5. Applies the training scaler to test data.

### Source Columns

Five generated numerical features.

### Incorrect Feature Transformation

The scaler learns mean and standard deviation from both training and testing observations.

### Correct Feature Transformation

The scaler learns mean and standard deviation from training observations only.

### Information Represented

The standardized values represent scaled numerical input features.

### ML Usefulness

Standardization helps models that are sensitive to feature scales.

### Leakage Risk

Fitting the scaler on the entire dataset introduces test-distribution information into preprocessing.

Important: This is a form of preprocessing leakage, even though it does not directly copy the target.

## Step 4 – Justify Your Feature

### Feature Name
Standardized Numerical Features.

### Source Columns
Five original numerical input features.

### Feature Logic / Formula

Standardized Value = (X - Mean) / Standard Deviation.

### Reason for Creation
To place numerical features on comparable scales.

### Expected Relationship with Target
Standardization changes feature scales but does not intentionally introduce target information.

### Potential Business Meaning
The features represent normalized versions of the original measurements.

### Potential ML Value
Helps scale-sensitive algorithms such as Logistic Regression, KNN, and SVM.

### Leakage Risk
High if the scaler is fitted on the complete dataset.

### Final Decision
RETAIN correctly standardized features using training-only fitted parameters.

# **Topic 4 – Temporal Leakage**

## Step 1 – Understand the Concept

Temporal Leakage occurs when a model uses information from the future to predict an event in the past or present.

It is especially common in time-dependent datasets.

### Example

A company wants to predict tomorrow's sales using today's available information.

If the model uses tomorrow's actual sales, future information has leaked into the input features.

### Common Causes

- Randomly splitting time-series data.
- Using future observations.
- Calculating rolling averages incorrectly.
- Using future transaction information.
- Creating features without considering prediction time.

### Important Principle

Every feature must be calculated using information available at or before the prediction time.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

Predicting tomorrow's temperature using tomorrow's actual temperature creates leakage.

### 2. Real-World Example

A hospital predicts patient deterioration using laboratory results collected after deterioration occurred.

### 3. Business Example

An e-commerce company predicts next month's sales using next month's actual sales.

### 4. AI/ML Use Case

A stock prediction model uses future stock prices when creating training features.

### Why is this useful?

Correct time-based feature engineering ensures that a model learns patterns using information that would actually be available at prediction time.

In [4]:
# Create daily sales dataset

dates = pd.date_range(
    start="2026-01-01",
    periods=30,
    freq="D"
)

sales_df = pd.DataFrame({
    "Date": dates,
    "Sales": np.random.randint(100, 1000, 30)
})

# Incorrect feature: next day's actual sales

sales_df["Future_Sales_Wrong"] = (
    sales_df["Sales"].shift(-1)
)

# Correct feature: previous day's sales

sales_df["Previous_Day_Sales"] = (
    sales_df["Sales"].shift(1)
)

# Display

display(sales_df.head(10))

,Date,Sales,Future_Sales_Wrong,Previous_Day_Sales
0,2026-01-01,636,360.0,NaN
1,2026-01-02,360,494.0,636.0
2,2026-01-03,494,325.0,360.0
3,2026-01-04,325,223.0,494.0
4,2026-01-05,223,137.0,325.0
5,2026-01-06,137,427.0,223.0
6,2026-01-07,427,350.0,137.0
7,2026-01-08,350,275.0,427.0
8,2026-01-09,275,701.0,350.0
9,2026-01-10,701,553.0,275.0


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates 30 days of sales observations.
2. Generates a future sales feature using shift(-1).
3. Generates a historical sales feature using shift(1).
4. Compares both feature columns.

### Incorrect Feature

Future_Sales_Wrong.

### Source Column

Sales.

### Incorrect Feature Logic

Future_Sales_Wrong = Next Day's Actual Sales.

### Correct Feature

Previous_Day_Sales.

### Correct Feature Logic

Previous_Day_Sales = Previous Day's Sales.

### Information Represented

Future_Sales_Wrong contains future information.

Previous_Day_Sales contains historical information.

### ML Usefulness

Previous_Day_Sales can be useful for forecasting future sales.

### Important Note

The correct feature must still be aligned with the prediction timestamp. Missing historical values should be handled appropriately.

## Step 4 – Justify Your Feature

### Feature Name
Previous_Day_Sales.

### Source Columns
Sales and Date.

### Feature Logic / Formula
Previous_Day_Sales = Sales.shift(1).

### Reason for Creation
To capture historical sales behaviour.

### Expected Relationship with Target
Previous-day sales may have a relationship with current or next-day sales.

### Potential Business Meaning
Represents the previous day's business sales performance.

### Potential ML Value
Can help a forecasting model learn sales trends.

### Leakage Risk
Low when correctly aligned with prediction time. Using shift(-1) would introduce future information.

### Final Decision
RETAIN Previous_Day_Sales.

REMOVE Future_Sales_Wrong.

# **Topic 5 – Post-Outcome Features**

## Step 1 – Understand the Concept

Post-Outcome Features are features that become available only after the event being predicted has already occurred.

Using these features during prediction creates leakage.

### Example

A bank wants to predict whether a loan will default.

Available information:
- Income
- Credit Score
- Loan Amount

Post-outcome information:
- Recovery Amount
- Collection Status
- Default Resolution Date

The post-outcome features cannot be used to predict the original default event.

### Important Principle

A feature must be available at the actual prediction time, not simply present in the dataset.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

Predicting exam results using marks awarded after the examination.

### 2. Real-World Example

Predicting hospital readmission using medical information collected after the patient's readmission.

### 3. Business Example

Predicting order delivery delays using the actual delivery date.

### 4. AI/ML Use Case

A loan default model uses the final recovery amount to predict whether the loan will default.

### Why is this useful?

It prevents a model from using information that is unavailable when predictions are actually made.

In [5]:
# Create loan prediction dataset

np.random.seed(42)

loan_data = pd.DataFrame({
    "Income": np.random.randint(20000, 100000, 500),
    "Credit_Score": np.random.randint(300, 850, 500),
    "Loan_Amount": np.random.randint(10000, 500000, 500)
})

# Create target

loan_data["Default"] = (
    (loan_data["Credit_Score"] < 550) &
    (loan_data["Loan_Amount"] > 250000)
).astype(int)

# Create post-outcome feature

loan_data["Recovery_Amount"] = np.where(
    loan_data["Default"] == 1,
    loan_data["Loan_Amount"] * 0.5,
    0
)

# Incorrect feature set

wrong_features = loan_data[
    ["Income", "Credit_Score",
     "Loan_Amount", "Recovery_Amount"]
]

# Correct feature set

correct_features = loan_data[
    ["Income", "Credit_Score", "Loan_Amount"]
]

print("Incorrect Features:")
display(wrong_features.head())

print("Correct Features:")
display(correct_features.head())

Incorrect Features:


,Income,Credit_Score,Loan_Amount,Recovery_Amount
0,35795,443,484460,242230.0
1,20860,338,119575,0.0
2,96820,408,383048,191524.0
3,74886,341,217606,0.0
4,26265,485,458322,229161.0


Correct Features:


,Income,Credit_Score,Loan_Amount
0,35795,443,484460
1,20860,338,119575
2,96820,408,383048
3,74886,341,217606
4,26265,485,458322


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates a loan dataset.
2. Generates a default target.
3. Creates Recovery_Amount using the default outcome.
4. Includes Recovery_Amount in an incorrect feature set.
5. Removes the post-outcome feature from the correct feature set.

### Incorrect Feature

Recovery_Amount.

### Source Columns

Default and Loan_Amount.

### Feature Logic

Recovery_Amount = 50% of Loan_Amount for defaulted loans.

### Information Represented

The amount recovered after the loan default event.

### Correct Features

- Income
- Credit_Score
- Loan_Amount

### ML Usefulness

The correct features represent information available before the loan outcome.

### Important Observation

Recovery_Amount may be useful for a different prediction task, such as predicting future recovery performance after default. Its validity depends on the prediction timestamp.

## Step 4 – Justify Your Feature

### Feature Name
Recovery_Amount.

### Source Columns
Default and Loan_Amount.

### Feature Logic / Formula
Recovery_Amount = Loan_Amount × 0.5 when Default = 1; otherwise 0.

### Reason for Creation
To demonstrate a feature created using post-outcome information.

### Expected Relationship with Target
It directly depends on the default outcome in this synthetic example.

### Potential Business Meaning
Represents money recovered after a loan default.

### Potential ML Value
Could be useful for post-default recovery analysis, but not for predicting the original default event.

### Leakage Risk
Very high for predicting the original default outcome.

### Final Decision
REMOVE for pre-default prediction.

# **Topic 6 – Leakage Through Aggregations**

## Step 1 – Understand the Concept

Aggregation is the process of summarizing multiple observations using calculations such as:

- Mean
- Sum
- Count
- Maximum
- Minimum
- Standard Deviation

Aggregation leakage occurs when these calculations include information that would not be available at the prediction time.

### Example

A company wants to predict whether a customer will make a future purchase.

If the customer's average spending is calculated using all transactions, including future transactions, leakage occurs.

### Incorrect Approach

Calculate customer averages using the complete dataset.

### Correct Approach

Calculate customer averages using only transactions available before the prediction time.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

A student's average score is calculated using marks from future examinations.

### 2. Real-World Example

A hospital calculates patient average test values using measurements collected after the prediction date.

### 3. Business Example

An e-commerce company calculates customer spending using transactions that occurred after the churn prediction date.

### 4. AI/ML Use Case

A recommendation system uses future customer purchase history when calculating historical spending features.

### Why is this useful?

Correct aggregation preserves the historical information available at prediction time and prevents future data from influencing the model.

In [6]:
# Create customer transaction data

transactions = pd.DataFrame({
    "Customer_ID": [101, 101, 101, 102, 102, 102],
    "Transaction_Date": pd.to_datetime([
        "2026-01-01",
        "2026-01-05",
        "2026-01-15",
        "2026-01-02",
        "2026-01-08",
        "2026-01-20"
    ]),
    "Amount": [1000, 2000, 5000, 1500, 2500, 7000]
})

# Prediction cutoff

cutoff_date = pd.Timestamp("2026-01-10")

# INCORRECT: Use all transactions

wrong_aggregation = (
    transactions.groupby("Customer_ID")["Amount"]
    .mean()
    .rename("Average_Spending_Wrong")
)

# CORRECT: Use historical transactions only

historical_transactions = transactions[
    transactions["Transaction_Date"] < cutoff_date
]

correct_aggregation = (
    historical_transactions.groupby("Customer_ID")["Amount"]
    .mean()
    .rename("Average_Spending_Correct")
)

# Compare

aggregation_comparison = pd.concat(
    [wrong_aggregation, correct_aggregation],
    axis=1
)

display(aggregation_comparison)

,Average_Spending_Wrong,Average_Spending_Correct
Customer_ID,,
101,2666.666667,1500.0
102,3666.666667,2000.0


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates customer transaction records.
2. Defines a prediction cutoff date.
3. Calculates average spending using all transactions.
4. Calculates average spending using historical transactions only.
5. Compares the two results.

### Incorrect Feature

Average_Spending_Wrong.

### Source Columns

Customer_ID, Transaction_Date, Amount.

### Incorrect Feature Logic

Average of all customer transactions, including transactions after the cutoff date.

### Correct Feature

Average_Spending_Correct.

### Correct Feature Logic

Average spending using only transactions before the cutoff date.

### Information Represented

Historical customer spending behaviour.

### ML Usefulness

Historical spending may help identify customer purchasing patterns and predict future behaviour.

### Important Observation

The prediction cutoff must be respected for every aggregation.

## Step 4 – Justify Your Feature

### Feature Name
Average_Spending_Correct.

### Source Columns
Customer_ID, Transaction_Date, Amount.

### Feature Logic / Formula
Mean of transaction amounts recorded before the prediction cutoff date.

### Reason for Creation
To summarize historical customer spending behaviour.

### Expected Relationship with Target
Higher or lower historical spending may be associated with future purchase behaviour or churn, depending on the dataset.

### Potential Business Meaning
Represents average historical customer spending.

### Potential ML Value
Helps a model understand customer purchasing patterns.

### Leakage Risk
Low when only historical transactions are included. High when future transactions are included.

### Final Decision
RETAIN Average_Spending_Correct.

REMOVE Average_Spending_Wrong for historical prediction tasks.

# **Topic 7 – Leakage Through Target Encoding**

## Step 1 – Understand the Concept

Target Encoding is a categorical feature encoding technique that replaces category values with a statistic calculated from the target variable.

For example, a product category can be replaced with its average purchase conversion rate.

Target Encoding can be useful for categorical features with many unique values.

However, incorrect target encoding can cause leakage.

### Incorrect Approach

Calculate category target averages using the complete dataset before splitting.

### Correct Approach

Calculate encoding values using training data only.

For training data, use out-of-fold or leave-one-out encoding to avoid using a row's own target to encode itself.

For testing and future data, use mappings learned from training data only.

### Important Principle

Target encoding must never use test targets to create training or test features.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

A student dataset contains different school names.

Replacing school names with average final marks using the entire dataset can introduce leakage.

### 2. Real-World Example

An online store encodes product categories using conversion rates calculated from both training and testing data.

### 3. Business Example

A bank encodes customer occupation using average default rates.

If testing outcomes are included, leakage occurs.

### 4. AI/ML Use Case

A fraud detection model uses merchant target encoding.

If future fraud labels influence encoding values, the model receives information from future outcomes.

### Why is this useful?

Correct target encoding captures category-level patterns while preserving independent model evaluation.

In [7]:
from sklearn.model_selection import train_test_split, KFold

# Create categorical dataset

np.random.seed(42)

target_df = pd.DataFrame({
    "Category": np.random.choice(
        ["A", "B", "C"],
        300
    )
})

# Create target with category-dependent probabilities

probabilities = target_df["Category"].map({
    "A": 0.2,
    "B": 0.5,
    "C": 0.8
})

target_df["Target"] = (
    np.random.rand(300) < probabilities
).astype(int)

# Split dataset

train_df, test_df = train_test_split(
    target_df,
    test_size=0.2,
    random_state=42,
    stratify=target_df["Target"]
)

# INCORRECT: Full dataset target encoding

wrong_mapping = target_df.groupby(
    "Category"
)["Target"].mean()

train_df["Wrong_Encoding"] = (
    train_df["Category"].map(wrong_mapping)
)

# CORRECT: Training-only mapping

correct_mapping = train_df.groupby(
    "Category"
)["Target"].mean()

global_mean = train_df["Target"].mean()

train_df["Correct_Encoding"] = (
    train_df["Category"]
    .map(correct_mapping)
    .fillna(global_mean)
)

test_df["Correct_Encoding"] = (
    test_df["Category"]
    .map(correct_mapping)
    .fillna(global_mean)
)

display(train_df.head())

display(test_df.head())

,Category,Target,Wrong_Encoding,Correct_Encoding
250,C,1,0.785047,0.770115
241,A,0,0.131313,0.141026
278,A,0,0.131313,0.141026
264,B,0,0.585106,0.586667
34,C,1,0.785047,0.770115


,Category,Target,Correct_Encoding
151,A,0,0.141026
131,A,0,0.141026
218,A,0,0.141026
129,B,0,0.586667
4,A,0,0.141026


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates categorical customer data.
2. Generates a binary target.
3. Splits the dataset into training and testing sets.
4. Calculates incorrect target encoding using the complete dataset.
5. Calculates correct category averages using training data only.
6. Applies the training mapping to testing data.
7. Uses the training global mean for unseen categories.

### Incorrect Feature

Wrong_Encoding.

### Source Columns

Category and Target.

### Incorrect Feature Logic

Category mean calculated using training and testing target values.

### Correct Feature

Correct_Encoding.

### Correct Feature Logic

Category mean calculated using training targets only.

### Information Represented

The average target value associated with each category.

### ML Usefulness

Target encoding converts categorical information into numerical representations.

### Important Observation

The training-only mapping prevents test-target leakage, but using the same training data to encode its own rows can still cause overfitting or self-target leakage.

For production-quality training features, use out-of-fold target encoding.

## Step 4 – Justify Your Feature

### Feature Name
Correct_Encoding.

### Source Columns
Category and training Target.

### Feature Logic / Formula
Category Encoding = Mean Target Value for that Category in Training Data.

### Reason for Creation
To represent categorical information using target-related statistics.

### Expected Relationship with Target
The encoding represents historical category-level target rates.

### Potential Business Meaning
Shows the historical average outcome associated with each category.

### Potential ML Value
Helps Machine Learning models use high-cardinality categorical features.

### Leakage Risk
High if test targets or a row's own target influence its encoding.

### Final Decision
RETAIN only with leakage-safe out-of-fold training encoding and training-only mappings for validation and test data.

# **Topic 8 – Detecting Leakage**

## Step 1 – Understand the Concept

Detecting Leakage means identifying features or preprocessing operations that provide information unavailable at prediction time.

Leakage can be difficult to identify because the model may show unusually good performance.

### Common Warning Signs

1. Extremely high model accuracy.
2. A feature has suspiciously high correlation with the target.
3. Features contain post-outcome information.
4. Model performance drops significantly in production.
5. Future information appears in historical features.
6. Train and test preprocessing is performed incorrectly.

### Detection Techniques

- Review feature definitions.
- Check prediction timestamps.
- Inspect feature-target relationships.
- Compare training and testing performance.
- Use time-based validation.
- Review data preprocessing pipelines.

Important: High correlation or high accuracy alone does not prove leakage. Domain knowledge and data timing are essential.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

A student marks prediction model achieves 100% accuracy because final marks are included as an input feature.

### 2. Real-World Example

A medical model achieves extremely high accuracy because diagnosis-related information was included in its input data.

### 3. Business Example

A sales prediction model performs unusually well because future sales information is included in historical features.

### 4. AI/ML Use Case

A churn prediction model includes cancellation date or final cancellation status.

Feature inspection can reveal suspicious columns.

### Why is this useful?

Detecting leakage helps identify invalid model inputs before deployment.

In [8]:
# Create a dataset with a suspicious feature

np.random.seed(42)

leakage_df = pd.DataFrame({
    "Age": np.random.randint(18, 65, 500),
    "Income": np.random.randint(20000, 100000, 500)
})

# Create target

leakage_df["Churn"] = (
    (leakage_df["Age"] < 30) &
    (leakage_df["Income"] < 40000)
).astype(int)

# Create leakage feature

leakage_df["Churn_Copy"] = leakage_df["Churn"]

# Check correlation

correlation = leakage_df.corr(numeric_only=True)

display(correlation)

# Identify perfect correlation with target

target_correlation = correlation["Churn"].drop("Churn")

suspicious_features = target_correlation[
    target_correlation.abs() == 1
]

print("Suspicious Features:")

display(suspicious_features)

# Check duplicate columns

print(
    "Churn_Copy equals Churn:",
    leakage_df["Churn_Copy"].equals(
        leakage_df["Churn"]
    )
)

,Age,Income,Churn,Churn_Copy
Age,1.000000,0.031812,-0.353017,-0.353017
Income,0.031812,1.000000,-0.307000,-0.307000
Churn,-0.353017,-0.307000,1.000000,1.000000
Churn_Copy,-0.353017,-0.307000,1.000000,1.000000


Suspicious Features:


Churn_Copy    1.0
Name: Churn, dtype: float64

Churn_Copy equals Churn: True


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates a customer dataset.
2. Generates a churn target.
3. Creates a copied target feature.
4. Calculates numerical correlations.
5. Identifies features with perfect target correlation.
6. Checks whether a suspicious feature exactly matches the target.

### Source Columns

Age, Income, Churn.

### Suspicious Feature

Churn_Copy.

### Feature Logic

Churn_Copy = Churn.

### Information Represented

The actual churn outcome.

### ML Usefulness

The copied feature provides no legitimate predictive value.

### Important Observation

Automated checks can identify suspicious patterns, but they cannot identify every type of leakage.

For example, temporal leakage may occur even when feature-target correlation is low.

## Step 4 – Justify Your Feature

### Feature Name
Churn_Copy.

### Source Columns
Churn.

### Feature Logic / Formula
Direct copy of the target column.

### Reason for Creation
To test whether leakage detection methods can identify a suspicious feature.

### Expected Relationship with Target
Perfect correlation.

### Potential Business Meaning
Contains the actual churn outcome.

### Potential ML Value
No valid predictive value.

### Leakage Risk
Extremely high.

### Final Decision
REMOVE.

Additional recommendation:

Review feature timestamps, source columns, data-generation logic, and model performance before approving any engineered feature.

# **Topic 9 – Preventing Leakage**

## Step 1 – Understand the Concept

Preventing Leakage means designing the Machine Learning workflow so that the model only learns from information that is legitimately available at prediction time.

### Best Practices

1. Define the prediction timestamp.
2. Understand when each feature becomes available.
3. Remove post-outcome features.
4. Split data before fitting preprocessing operations.
5. Use training-only scaling and imputation.
6. Apply PCA only after training data is separated.
7. Use time-based splitting for temporal problems.
8. Apply leakage-safe target encoding.
9. Use pipelines for preprocessing and model training.
10. Validate using realistic evaluation data.

### Important Principle

The training process must reproduce the information conditions of real-world prediction.

## Step 2 – Demonstrate the Concept

### 1. Simple Example

A student marks prediction model should use study hours and attendance, not final exam marks.

### 2. Real-World Example

A medical model should only use patient information available before the diagnosis prediction time.

### 3. Business Example

An e-commerce company should use historical transactions to predict future customer purchases.

### 4. AI/ML Use Case

A churn prediction model should use historical customer information and avoid actual cancellation details.

### Why is this useful?

It creates reliable training and evaluation workflows and helps models perform more realistically in production.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create a clean dataset

np.random.seed(42)

safe_df = pd.DataFrame({
    "Age": np.random.randint(18, 65, 500),
    "Income": np.random.randint(20000, 100000, 500),
    "Tenure": np.random.randint(1, 60, 500)
})

# Create target

safe_df["Churn"] = (
    (safe_df["Tenure"] < 12) &
    (safe_df["Income"] < 40000)
).astype(int)

# Separate features and target

X = safe_df[
    ["Age", "Income", "Tenure"]
]

y = safe_df["Churn"]

# Split before preprocessing

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create leakage-safe pipeline

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Train pipeline

pipeline.fit(X_train, y_train)

# Predict

predictions = pipeline.predict(X_test)

# Evaluate

print("Test Accuracy:",
      accuracy_score(y_test, predictions))

print("Training Rows:", len(X_train))

print("Testing Rows:", len(X_test))

Test Accuracy: 0.97
Training Rows: 400
Testing Rows: 100


## Step 3 – Implement the Concept: Code Explanation

### What does the code do?

1. Creates a customer dataset.
2. Generates a churn target.
3. Separates input features and target.
4. Splits the dataset before preprocessing.
5. Creates a Scikit-learn Pipeline.
6. Imputes missing values using training data.
7. Standardizes numerical features using training data.
8. Trains Logistic Regression.
9. Evaluates the model on unseen testing data.

### Source Columns

Age, Income, Tenure.

### Feature Logic

The original input features are used without adding target-derived columns.

### Why were these features selected?

They represent customer information that is intended to be available before the churn outcome.

### Information Represented

- Age: Customer age.
- Income: Customer income.
- Tenure: Duration of the customer relationship.

### ML Usefulness

These features help the model learn customer characteristics and potential churn patterns.

### Leakage Prevention

The pipeline ensures that imputation and scaling are fitted on training data only.

The testing dataset is only transformed using the fitted preprocessing steps.

## Step 4 – Justify Your Feature

### Feature Name
Age, Income, Tenure.

### Source Columns
Original customer dataset columns.

### Feature Logic / Formula
Original numerical feature values, with missing values imputed and features standardized using training data.

### Reason for Creation
To provide legitimate customer information for churn prediction.

### Expected Relationship with Target
Tenure and Income may have relationships with customer churn in this synthetic dataset. Age may provide additional information depending on the data.

### Potential Business Meaning
Represents customer demographics, financial characteristics, and relationship duration.

### Potential ML Value
Helps the model identify customer patterns associated with churn.

### Leakage Risk
Low, provided the columns are genuinely available at prediction time.

### Final Decision
RETAIN all three candidate features and validate their usefulness using appropriate evaluation metrics.